# 03 - Build aggregates and sync to Rayfin SQL (v2)
POC 31: CineScope. Builds the analytical tables from `raw_titles` and
`raw_credits`, then performs a full-refresh load into the Rayfin-provisioned
SQL database. Designed to be run end to end with Run All after setting the
two config values below.

**Load design - why JSON, not parameter binding:**
rows are serialised to JSON and inserted with `INSERT ... SELECT FROM
OPENJSON(?) WITH (...)`, where the column types in the WITH clause are read
from `INFORMATION_SCHEMA.COLUMNS` of the actual target table. The database
does all type conversion server-side. This eliminates the entire class of
pyodbc/pandas binding failures (nullable ints upcast to float, numpy
scalars, sql_variant quirks) hit with executemany during the first build.

**Connection facts (verified against a live Fabric App, June 2026):**
- Use the App's child SQL database WRITABLE endpoint:
  `*.database.fabric.microsoft.com`. The SQL analytics endpoint
  (`*.datawarehouse.fabric.microsoft.com`) accepts connections and
  metadata queries but rejects all DML with error 24559
- The catalog name carries a generated GUID suffix, e.g.
  `poc-31-cinescope-tmdb-3b4be5dd-...`. Copy it verbatim from the
  connection string (portal: SQL database > Settings > Connection strings)
- Entra token audience `https://database.windows.net/.default` works for
  the notebook identity via `notebookutils`
- Table names are resolved from INFORMATION_SCHEMA at run time - Rayfin
  generates the schema and names may not match entity class names

**Full-refresh semantics:** all five tables are deleted (children first)
and reinserted. Consistent with the 6-month TMDB refresh obligation and
idempotent at the run level.

**Prerequisites:** Notebooks 01 and 02 completed; `npx rayfin up db apply`
has created the five tables.


In [ ]:
# Configuration - the only cell you edit
# SQL_SERVER: writable endpoint host ONLY (no port, no prefix). From the
#   Fabric App's child SQL database > Settings > Connection strings >
#   Data Source, WITHOUT the ",1433".
# SQL_DATABASE: the Initial Catalog value VERBATIM (includes GUID suffix).

SQL_SERVER = ""    # e.g. xxxx-yyyy.database.fabric.microsoft.com
SQL_DATABASE = ""  # e.g. poc-31-cinescope-tmdb-3b4be5dd-...

assert SQL_SERVER and SQL_DATABASE, "Set SQL_SERVER and SQL_DATABASE first"
assert "datawarehouse" not in SQL_SERVER, "That is the read-only analytics endpoint - use *.database.fabric.microsoft.com"


In [ ]:
# Build dataframes from the lakehouse Delta tables
import uuid
import pandas as pd

NS = uuid.UUID("6ba7b810-9dad-11d1-80b4-00c04fd430c8")  # fixed namespace

def stable_id(*parts):
    return str(uuid.uuid5(NS, ":".join(str(p) for p in parts)))

titles = spark.table("raw_titles").toPandas()
credits = spark.table("raw_credits").toPandas()

titles["id"] = titles["tmdbKey"].map(lambda k: stable_id("title", k))
credits["person_uuid"] = credits["personTmdbId"].map(lambda i: stable_id("person", i))
credits["title_uuid"] = credits["titleKey"].map(lambda k: stable_id("title", k))
credits["principal_id"] = credits.apply(
    lambda r: stable_id("principal", r["titleKey"], r["personTmdbId"], r["category"]), axis=1)
credits = credits.drop_duplicates(subset=["principal_id"])
credits = credits[credits["title_uuid"].isin(set(titles["id"]))]
print(f"{len(titles):,} titles, {len(credits):,} credits")


In [ ]:
# Person career stats (vote-weighted)
joined = credits.merge(
    titles[["id", "voteAverage", "voteCount"]],
    left_on="title_uuid", right_on="id", suffixes=("", "_t"))

def person_stats(g):
    votes = g["voteCount"].sum()
    avg = (g["voteAverage"] * g["voteCount"]).sum() / votes if votes else g["voteAverage"].mean()
    return pd.Series({
        "titleCount": g["titleKey"].nunique(),
        "avgRating": round(float(avg), 2),
        "totalVotes": int(votes),
        "dominantRole": "director" if (g["category"] == "director").mean() >= 0.5 else "cast",
    })

stats = joined.groupby("person_uuid").apply(person_stats).reset_index()
meta = credits.sort_values("ordering").drop_duplicates("person_uuid")[
    ["person_uuid", "personTmdbId", "personName", "knownForDepartment", "profilePath"]]
persons = meta.merge(stats, on="person_uuid")
print(f"{len(persons):,} people ({(persons['dominantRole']=='director').sum():,} directors)")


In [ ]:
# YearStat and GenreYearStat
def weighted(g):
    votes = g["voteCount"].sum()
    avg = (g["voteAverage"] * g["voteCount"]).sum() / votes if votes else g["voteAverage"].mean()
    return round(float(avg), 2), int(votes)

year_rows = []
for (year, mt), g in titles.groupby(["releaseYear", "mediaType"]):
    avg, votes = weighted(g)
    rt = g.loc[g["mediaType"] == "movie", "runtimeMinutes"].dropna()
    year_rows.append({
        "id": stable_id("yearstat", year, mt), "statKey": f"{year}-{mt}",
        "year": int(year), "mediaType": mt, "titleCount": int(len(g)),
        "avgRating": avg, "avgRuntime": round(float(rt.mean()), 1) if len(rt) else None,
        "totalVotes": votes})

exploded = titles.assign(genre=titles["genres"].str.split(",")).explode("genre")
exploded = exploded[exploded["genre"].astype(bool)]
genre_rows = []
for (genre, year, mt), g in exploded.groupby(["genre", "releaseYear", "mediaType"]):
    avg, votes = weighted(g)
    genre_rows.append({
        "id": stable_id("genrestat", genre, year, mt), "statKey": f"{genre}-{year}-{mt}",
        "genre": genre, "year": int(year), "mediaType": mt,
        "titleCount": int(len(g)), "avgRating": avg, "totalVotes": votes})

print(f"{len(year_rows):,} year stats, {len(genre_rows):,} genre-year stats")


In [ ]:
# Connect to the writable Rayfin SQL endpoint and verify
import struct, pyodbc
from notebookutils import credentials

token = credentials.getToken("https://database.windows.net/.default").encode("utf-16-le")
token_struct = struct.pack(f"<I{len(token)}s", len(token), token)

conn = pyodbc.connect(
    f"DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={SQL_SERVER},1433;"
    f"DATABASE={SQL_DATABASE};Encrypt=yes",
    attrs_before={1256: token_struct}, timeout=30)  # 1256 = SQL_COPT_SS_ACCESS_TOKEN
cur = conn.cursor()

# DATABASEPROPERTYEX returns sql_variant which pyodbc cannot map - CAST it
db, mode = cur.execute(
    "SELECT DB_NAME(), CAST(DATABASEPROPERTYEX(DB_NAME(), 'Updateability') AS varchar(20))"
).fetchone()
print(db, mode)
assert mode == "READ_WRITE", "Connected to a read-only endpoint - check SQL_SERVER"


In [ ]:
# Resolve generated table names and read their column types
_tables = cur.execute(
    "SELECT TABLE_SCHEMA, TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_TYPE='BASE TABLE'"
).fetchall()
print("Tables:", [f"{s}.{t}" for s, t in _tables])

# Rayfin pluralises table names, including irregular plurals
# (verified live: Title->Titles, Person->People, Principal->Principals,
# YearStat->YearStats, GenreYearStat->GenreYearStats)
TABLE_MAP = {
    "Title": "Titles",
    "Person": "People",
    "Principal": "Principals",
    "YearStat": "YearStats",
    "GenreYearStat": "GenreYearStats",
}
ENTITIES = tuple(TABLE_MAP)

def resolve(entity):
    target = TABLE_MAP[entity]
    c = [(s, t) for s, t in _tables if t == target]
    assert len(c) == 1, f"Expected table {target} for {entity}; found {c}; all: {_tables}"
    return c[0]

def column_types(schema, table):
    q = ("SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH, "
         "NUMERIC_PRECISION, NUMERIC_SCALE FROM INFORMATION_SCHEMA.COLUMNS "
         "WHERE TABLE_SCHEMA = ? AND TABLE_NAME = ?")
    out = {}
    for col, dt, clen, prec, scale in cur.execute(q, schema, table).fetchall():
        if dt in ("nvarchar", "varchar", "nchar", "char"):
            out[col] = f"{dt}(max)" if clen == -1 else f"{dt}({clen})"
        elif dt in ("decimal", "numeric"):
            out[col] = f"decimal({prec},{scale})"
        else:
            out[col] = dt
    return out

for e in ENTITIES:
    s, t = resolve(e)
    print(e, "->", f"[{s}].[{t}]")


In [ ]:
# Loader: JSON in, server-side typed extraction. No client-side binding.
import json as _json
import numpy as np

def py(v):
    if v is None or (isinstance(v, (float, np.floating)) and pd.isnull(v)):
        return None
    if isinstance(v, (np.bool_, bool)):
        return bool(v)
    if isinstance(v, np.integer):
        return int(v)
    if isinstance(v, (np.floating, float)):
        f = float(v)
        return int(f) if f.is_integer() else f
    return v

def records_of(df):
    cols = list(df.columns)
    return [dict(zip(cols, (py(v) for v in row)))
            for row in df.itertuples(index=False, name=None)]

def bulk_insert(entity, recs, batch=8000):
    schema, table = resolve(entity)
    types = column_types(schema, table)
    cols = [c for c in recs[0].keys()]
    missing = [c for c in cols if c not in types]
    assert not missing, f"{entity}: columns not in target table: {missing}"
    with_clause = ", ".join(f"[{c}] {types[c]} '$.{c}'" for c in cols)
    col_list = ", ".join(f"[{c}]" for c in cols)
    for i in range(0, len(recs), batch):
        payload = _json.dumps(recs[i:i + batch], ensure_ascii=False)
        cur.execute(
            f"INSERT INTO [{schema}].[{table}] ({col_list}) "
            f"SELECT {col_list} FROM OPENJSON(?) WITH ({with_clause})",
            payload)
    conn.commit()
    print(f"[{schema}].[{table}]: {len(recs):,} rows inserted")


In [ ]:
# Full refresh: delete children first, insert parents first
for e in ("Principal", "Title", "Person", "YearStat", "GenreYearStat"):
    s, t = resolve(e)
    cur.execute(f"DELETE FROM [{s}].[{t}]")
conn.commit()
print("Existing rows cleared")

title_df = titles[["id", "tmdbKey", "mediaType", "title", "releaseYear", "decade",
                   "runtimeMinutes", "genres", "voteAverage", "voteCount",
                   "popularity", "posterPath", "originalLanguage", "isSeries"]]

person_df = persons.rename(columns={
    "person_uuid": "id", "personTmdbId": "tmdbKey", "personName": "name"})[
    ["id", "tmdbKey", "name", "knownForDepartment", "profilePath",
     "dominantRole", "titleCount", "avgRating", "totalVotes"]]
person_df = person_df.assign(tmdbKey=person_df["tmdbKey"].astype(str))

principal_df = credits.rename(columns={
    "principal_id": "id", "title_uuid": "title_id", "person_uuid": "person_id"})[
    ["id", "title_id", "person_id", "category", "ordering", "characterName"]]
principal_df = principal_df[principal_df["person_id"].isin(set(person_df["id"]))]

bulk_insert("Title", records_of(title_df))
bulk_insert("Person", records_of(person_df))
bulk_insert("Principal", records_of(principal_df))
bulk_insert("YearStat", year_rows)
bulk_insert("GenreYearStat", genre_rows)

for e in ENTITIES:
    s, t = resolve(e)
    print(e, cur.execute(f"SELECT COUNT(*) FROM [{s}].[{t}]").fetchone()[0])
conn.close()


## Done
Open the deployed CineScope app (hosting URL from `npx rayfin up`) or run
`npx rayfin env && npm run dev` locally - all four views should render.

Diarise: TMDB caching terms require this data to be refreshed (re-run
notebooks 01-03) or deleted within 6 months of today's run.
